## Things to note
1. get data into pandas/numpy. Appy one-hot encoding per city. remove air_pollution_features_row_id. remove time of measurement. 
2. train test split must be chronological to prevent data leakage.
3. Use light gbm (classifier) to reduce training time. Note that classifier does not take into consideration that 1 and 2 are close predictions, so evaluation should take into consideration that (get both exact accuracy and within one accuracy scores)
4. evaluate model (also compare it against just guessing the 6 hour air pollution = current air pollution since they are highly correlated anyway. my model should outperform that) and save it to a file in minio

In [1]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import clickhouse_connect
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
client = clickhouse_connect.get_client(
    host='localhost',
    port=8143,
    username='default',
    password=os.getenv("CLICKHOUSE_PASSWORD")
)

In [42]:
air_pollution_features_df = client.query_df("SELECT * FROM weather_features.feature_air_pollution order by time_of_measurement")
air_pollution_features_df = air_pollution_features_df.dropna()

# Appy one-hot encoding per city and season. remove air_pollution_features_row_id. remove time of measurement. 
air_pollution_features_df = pd.get_dummies(air_pollution_features_df, columns=['lat_and_long_encoding_id'], dtype=int)
air_pollution_features_df = pd.get_dummies(air_pollution_features_df, columns=['season'], dtype=int)
air_pollution_features_df=air_pollution_features_df.drop(columns="air_pollution_features_row_id")
air_pollution_features_df=air_pollution_features_df.drop(columns="time_of_measurement") # relevant time features like day and hour and weekend or not are encoded already.

In [43]:
len(air_pollution_features_df)

187164

In [44]:
air_pollution_features_df.head(100)

,co_concentration,no_concentration,no2_concentration,o3_concentration,so2_concentration,pm2_5_concentration,pm10_concentration,nh3_concentration,aqi,aqi_1_hour_past,...,lat_and_long_encoding_id_69f0f0f1012c1bce555f7de22f849232,lat_and_long_encoding_id_6e26eb15061b34b2b95908f46f50c496,lat_and_long_encoding_id_dc07b90435445c5e64d7c73cbf766827,lat_and_long_encoding_id_de87d594f95f5454e6861e2026e664d0,season_Autumn,season_Dry Season,season_Rainy Season,season_Spring,season_Summer,season_Winter
1008,290.390015,0.000000,1.970000,50.779999,0.410000,94.059998,273.670013,1.71,5,5,...,0,0,0,0,0,1,0,0,0,0
1009,293.730011,0.000000,2.100000,57.220001,1.280000,11.880000,20.990000,6.84,2,2,...,1,0,0,0,0,1,0,0,0,0
1010,243.660004,2.490000,27.080000,0.460000,37.669998,9.110000,16.540001,1.22,2,2,...,0,0,1,0,0,0,0,0,1,0
1011,447.269989,0.000000,3.770000,32.900002,1.100000,21.410000,92.870003,2.53,3,3,...,0,0,0,0,0,1,0,0,0,0
1012,647.539978,50.509998,39.070000,0.000000,26.700001,34.369999,62.450001,3.45,3,3,...,0,1,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1103,1455.310059,6.200000,19.879999,45.060001,5.360000,23.440001,29.639999,3.64,2,2,...,0,0,0,1,0,1,0,0,0,0
1104,727.650024,4.860000,20.730000,50.070000,0.910000,38.680000,86.279999,1.38,3,3,...,0,0,0,0,0,1,0,0,0,0
1105,330.450012,0.000000,4.670000,70.099998,2.950000,14.010000,21.570000,6.02,2,2,...,1,0,0,0,0,1,0,0,0,0
1106,814.440002,19.219999,66.489998,10.460000,30.990000,25.639999,36.200001,7.41,3,2,...,0,0,1,0,0,0,0,0,1,0


In [45]:
# Count rows where you can guess the aqi 6 hours ahead because it is the same, as a comparison baseline
current_and_six_hour_ahead_same_fraction = ((air_pollution_features_df['aqi'] == air_pollution_features_df['aqi_6_hour_ahead']).sum())/len(air_pollution_features_df)
print(current_and_six_hour_ahead_same_fraction)

0.6088350323780214


In [46]:
Y=air_pollution_features_df["aqi_6_hour_ahead"]
X=air_pollution_features_df.drop(columns="aqi_6_hour_ahead")

# Slice chronologically using .iloc; 80-20 split
split_idx = int(len(air_pollution_features_df) * 0.7)
X_train,X_test = X.iloc[:split_idx],X.iloc[split_idx:]
Y_train,Y_test = Y.iloc[:split_idx],Y.iloc[split_idx:]

In [47]:
X_test

,co_concentration,no_concentration,no2_concentration,o3_concentration,so2_concentration,pm2_5_concentration,pm10_concentration,nh3_concentration,aqi,aqi_1_hour_past,...,lat_and_long_encoding_id_69f0f0f1012c1bce555f7de22f849232,lat_and_long_encoding_id_6e26eb15061b34b2b95908f46f50c496,lat_and_long_encoding_id_dc07b90435445c5e64d7c73cbf766827,lat_and_long_encoding_id_de87d594f95f5454e6861e2026e664d0,season_Autumn,season_Dry Season,season_Rainy Season,season_Spring,season_Summer,season_Winter
136318,134.360001,0.00,4.71,71.919998,13.44,24.260000,66.660004,3.30,3,3,...,0,1,0,0,0,0,0,0,1,0
136319,93.150002,0.00,0.32,42.400002,0.39,4.590000,19.990000,0.00,1,2,...,0,0,0,1,0,0,1,0,0,0
136320,303.589996,0.00,1.42,25.330000,0.33,5.200000,9.410000,0.40,1,1,...,0,0,0,0,0,0,1,0,0,0
136321,295.290009,0.01,1.22,10.540000,0.17,2.420000,3.190000,0.92,1,1,...,1,0,0,0,0,0,1,0,0,0
136322,337.359985,0.00,9.42,32.349998,1.48,23.790001,30.980000,2.23,2,2,...,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196687,141.710007,0.00,1.34,74.650002,0.45,18.379999,38.209999,0.89,2,2,...,1,0,0,0,0,0,1,0,0,0
196688,172.910004,0.00,4.52,83.580002,1.26,9.790000,13.120000,4.10,2,2,...,0,0,1,0,0,0,0,1,0,0
196689,148.539993,0.00,0.07,21.799999,0.05,1.250000,2.540000,0.01,1,1,...,0,0,0,0,0,0,1,0,0,0
196690,143.770004,0.00,4.44,111.650002,6.66,22.450001,54.310001,1.83,3,3,...,0,1,0,0,1,0,0,0,0,0


In [48]:
model=LGBMClassifier()
model.fit(X_train,Y_train,
         eval_set=[(X_train, Y_train), (X_test, Y_test)],
    eval_names=['train', 'eval'],
    eval_metric='multi_logloss')
Y_pred=model.predict(X_test)
acs=accuracy_score(Y_test,Y_pred)
print(acs*100)

C:\Users\sashe\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038369 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10794
[LightGBM] [Info] Number of data points in the train set: 131014, number of used features: 70
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -1.412949
[LightGBM] [Info] Start training from score -1.064605
[LightGBM] [Info] Start training from score -1.475341
[LightGBM] [Info] Start training from score -2.369533
[LightGBM] [Info] Start training from score -2.413666
78.03918076580588


In [49]:
train_error_history = model.evals_result_['eval']['multi_logloss']
final_train_error = train_error_history[-1]

print(f"Final Training Error: {final_train_error}")
# train_error_history

Final Training Error: 0.5221668890258546


In [50]:
# split randomly to see leakage exercise
from sklearn.model_selection import train_test_split

X_train_leakage, X_test_leakage, Y_train_leakage, Y_test_leakage = train_test_split(
    X, Y, test_size=0.30, random_state=2
)

model=LGBMClassifier()
model.fit(X_train_leakage,Y_train_leakage,
         eval_set=[(X_train_leakage,Y_train_leakage), (X_test_leakage, Y_test_leakage)],
    eval_names=['train', 'eval'],
    eval_metric='multi_logloss')
Y_pred_leakage=model.predict(X_test_leakage)
acs=accuracy_score(Y_test_leakage,Y_pred_leakage)
print(acs*100)

C:\Users\sashe\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034985 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10791
[LightGBM] [Info] Number of data points in the train set: 131014, number of used features: 70
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -1.250723
[LightGBM] [Info] Start training from score -1.059374
[LightGBM] [Info] Start training from score -1.507216
[LightGBM] [Info] Start training from score -2.561186
[LightGBM] [Info] Start training from score -2.683986
73.70436331255566


In [40]:
print(Y_train.mean())
print(Y_train.var())
print(Y_test.mean())
print(Y_test.var())

2.439399827781969
1.4573667748532029
1.950464066000498
0.9059937214465851
